In [10]:
import os

In [11]:
%pwd

'c:\\Users\\Sandeep\\Desktop\\Projects\\Text_Summariser\\reseach'

In [12]:
os.chdir("../")

In [13]:
%pwd

'c:\\Users\\Sandeep\\Desktop\\Projects\\Text_Summariser'

In [14]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen = True)
class ModelEvaluationConfig:

    root_dir: Path
    data_path: Path
    model_path: Path
    tokenizer_path: Path
    metric_file_name: Path

In [15]:
from Text_summariser.constant import *
from Text_summariser.utils.common import read_yaml, create_directories

In [16]:
# 4 Update configuration manager

class configurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,     # Access to constants
        params_filepath = PARAMS_FILE_PATH):

        self.config = read_yaml(config_filepath) # read all config and params yaml files
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root]) # same upto here for most pipeline

    def get_model_evaluation_config(self) -> ModelEvaluationConfig:
        config = self.config.model_evaluation

        create_directories([config.root_dir])

        model_evaluation_config = ModelEvaluationConfig(  # ← Capital M and E and C
            root_dir=config.root_dir,
            data_path=config.data_path,
            model_path=config.model_path,
            tokenizer_path=config.tokenizer_path,
            metric_file_name=config.metric_file_name
        )

        return model_evaluation_config

In [17]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from datasets import load_dataset, load_from_disk
import torch
import pandas as pd
from tqdm import tqdm
import evaluate                  
rouge = evaluate.load("rouge")                                                                  

In [18]:
# Conponents
class ModelEvaluation:
    def __init__(self, config: ModelEvaluationConfig):
        self.config = config

    def generate_batch_sized_chunk(self, list_of_elements, batch_size):
        for i in range(0, len(list_of_elements), batch_size):
            yield list_of_elements[i : i + batch_size]

    def calculate_metrics_on_test_ds(self, dataset, metric, model, tokenizer,
                                      batch_size=16,
                                      device="cuda" if torch.cuda.is_available() else "cpu",
                                      column_text="dialogue",
                                      column_summary="summary"):
        
        article_batches = list(self.generate_batch_sized_chunk(dataset[column_text], batch_size))
        target_batches  = list(self.generate_batch_sized_chunk(dataset[column_summary], batch_size))

        for article_batch, target_batch in tqdm(
            zip(article_batches, target_batches), total=len(article_batches)):

            inputs = tokenizer(
                article_batch,                  # ← was article_batches (whole list!)
                max_length=1024,
                truncation=True,
                padding="max_length",           # ← was max_lenght
                return_tensors="pt"             # ← was return_tensor
            )

            summaries = model.generate(
                input_ids=inputs["input_ids"].to(device),        # ← was input not inputs
                attention_mask=inputs["attention_mask"].to(device),
                length_penalty=0.8,             # ← was lenght_penalty
                num_beams=8,
                max_length=128
            )

            decoded_summaries = [
                tokenizer.decode(s,
                    skip_special_tokens=True,           # ← was skip_special_token
                    clean_up_tokenization_spaces=True)  # ← was clean_up_tokenization_sapce
                for s in summaries
            ]

            decoded_summaries = [d.replace("<n>", " ") for d in decoded_summaries]  # ← was replace("", " ")

            metric.add_batch(                   # ← was metrics (no s)
                predictions=decoded_summaries,  # ← was decoded_summariess and prediction
                references=target_batch         # ← was target_bactch and referenmce
            )

        score = metric.compute()
        return score

    def evaluate(self):                         # ← was evaulate
        device = "cuda" if torch.cuda.is_available() else "cpu"
        
        tokenizer = AutoTokenizer.from_pretrained(self.config.tokenizer_path)        # ← was frompretrainede
        model = AutoModelForSeq2SeqLM.from_pretrained(self.config.model_path).to(device)  # ← was Automodelforseq2seqlm

        # Load rouge — new way!
        rouge = evaluate.load("rouge")          # ← replaces load_metric

        # Load data
        dataset_samsum = load_from_disk(self.config.data_path)

        # Run evaluation
        score = self.calculate_metrics_on_test_ds(
            dataset=dataset_samsum["test"],
            metric=rouge,
            model=model,
            tokenizer=tokenizer,
            batch_size=16,
            device=device,
            column_text="dialogue",             # ← SAMSum column names
            column_summary="summary"
        )

        # Create the correct directory
        os.makedirs(os.path.dirname(self.config.metric_file_name), exist_ok=True)

        # Save metrics
        df = pd.DataFrame([score], index=["rouge_score"])

        df.to_csv(
            self.config.metric_file_name,
            index=False
        )

        print("Evaluation scores:", score)

In [21]:
from Text_summariser.logging import logger

In [22]:
try:
    config = configurationManager()
    model_evaluation_config = config.get_model_evaluation_config()
    
    if not os.path.exists(model_evaluation_config.metric_file_name):
        logger.info("No metrics found — running evaluation...")
        model_evaluation = ModelEvaluation(config=model_evaluation_config)
        model_evaluation.evaluate()
    else:
        logger.info("Metrics already exist — skipping evaluation!")

except Exception as e:
    raise e

[2026-05-16 00:34:49,484: INFO: common: ymal file config\config.yaml loaded sucessfully]
[2026-05-16 00:34:49,487: INFO: common: ymal file params.yaml loaded sucessfully]
[2026-05-16 00:34:49,490: INFO: common: created directory at artifacts]
[2026-05-16 00:34:49,492: INFO: common: created directory at artifacts/model_trainer]
[2026-05-16 00:34:49,493: INFO: 2945904049: Metrics already exist — skipping evaluation!]
